# BGDY 3x3 · configuraciones *rattled* → inputs de QE (SCF)

Parte de la lámina relajada `relax_final.xyz` (126 átomos, **ya es la 3x3**: no hay que repetir la celda) y genera
200 configuraciones con el *rattle* Monte Carlo de hiPhive. Por cada una escribe `cases/case_XXX/input_scf.in`.

**Rattle** = desplazar al azar cada átomo alrededor de su posición relajada. Así se obtienen estructuras fuera del
mínimo, con energías y fuerzas distintas de cero: justo lo que necesita un dataset para entrenar un potencial.

Necesita `ase` y `hiphive` (`pip install ase hiphive`). Tarda unos minutos: casi todo el tiempo es hiPhive.

In [ ]:
!pip install hiphive

## 1. Parámetros

In [ ]:
import numpy as np
from pathlib import Path
from ase.io import read, write
from ase.neighborlist import neighbor_list
from hiphive.structure_generation import generate_mc_rattled_structures

REF  = "relax_final.xyz"                 # lámina 3x3 relajada
STDS = [0.005, 0.01, 0.02, 0.03, 0.04]   # amplitudes de rattle (Å): una tanda por valor
N    = 40                                # configuraciones por amplitud  →  5 × 40 = 200
SEED = 42

D_MIN_FACTOR    = 0.93   # d_min de hiPhive = 0.93 · distancia mínima de la referencia (como en tu ejemplo)
MIN_DIST_FACTOR = 0.90   # filtro duro: se descarta toda config con algún par más cerca de 0.90 · d_ref

**Sobre las amplitudes.** En hiPhive el desplazamiento real **no** es `rattle_std`: cada átomo se mueve ~10 veces
(`n_iter=10`), así que el desplazamiento típico es ≈ √10 · `rattle_std`. Con estos valores, |u| rms ≈ 0.03 – 0.21 Å.
No subas de ~0.04: con 0.05 el triple enlace C≡C (1.23 Å) llegaba a comprimirse hasta 1.01 Å.

## 2. Input de QE

La cabecera del `input_scf.in`. Son los parámetros de tu template NVT (`ecutrho=600`, `degauss=0.005`,
`conv_thr=7.35d-6`) con `calculation='scf'`. Detalles:

- `{nat}` se rellena en el código; la celda y las posiciones se añaden después, igual que en tu workflow de vacantes.
- `pseudo_dir='../../pseudos'`: `pw.x` corre dentro de `cases/case_XXX/`, así que los UPF se leen de `pseudos/`
  sin copiarlos a cada carpeta.
- `disk_io='nowf'`: no guarda las funciones de onda (un SCF no se reanuda, se repite).
- `tprnfor` y `tstress`: necesarios para tener fuerzas y tensión en el dataset.

In [ ]:
HEAD = """&CONTROL
 calculation = 'scf'
 prefix = 'bgdy'
 pseudo_dir = '../../pseudos'
 outdir = './save'
 disk_io = 'nowf'
 tprnfor = .true.
 tstress = .true.
/
&SYSTEM
 ibrav = 0
 nat = {nat}
 ntyp = 2
 nspin = 2
 starting_magnetization(1) = 0.1d0
 starting_magnetization(2) = 0.1d0
 ecutwfc = 60.0
 ecutrho = 540.0
 vdw_corr = 'Grimme-D3'
 occupations = 'smearing'
 smearing = 'gaussian'
 degauss = 0.005
 nosym = .true.
/
&ELECTRONS
 electron_maxstep = 250
 mixing_mode = 'plain'
 mixing_beta = 0.2
 conv_thr = 7.35d-6
/
ATOMIC_SPECIES
C 12.011 C.pbe-n-kjpaw_psl.1.0.0.UPF
B 10.811 B.pbe-n-kjpaw_psl.1.0.0.UPF
"""

## 3. Generar el rattle

Es tu `generate_rattled_structures`, adaptada a una lámina 2D:

1. `d0` = distancia mínima entre átomos de la referencia (el triple enlace C≡C, ≈ 1.23 Å).
2. `generate_mc_rattled_structures` mueve los átomos por Monte Carlo y **penaliza** (no prohíbe) los movimientos que
   dejan algún par por debajo de `d_min = 0.93 · d0`.
3. Como la penalización es probabilística, añadimos un **filtro duro**: se descartan las configuraciones con algún par
   a menos de `0.90 · d0` y se generan más (con otra semilla) hasta completar `N`. Con amplitudes altas se descarta
   más de la mitad; es normal.
4. Se imprime, por amplitud, el desplazamiento rms/máximo y la distancia mínima resultante: **compruébalos**.

In [ ]:
ref = read(REF)

def dmin(atoms):
    """Distancia mínima entre pares de átomos (con condiciones periódicas)."""
    return neighbor_list("d", atoms, 3.0).min()

d0 = dmin(ref)
configs = []                                           # lista de (rattle_std, Atoms)

for k, std in enumerate(STDS):
    good = []
    for t in range(50):                                # repone las descartadas con otra semilla
        batch = generate_mc_rattled_structures(ref, N, std, D_MIN_FACTOR * d0, seed=SEED + 100 * k + t)
        good += [a for a in batch if dmin(a) >= MIN_DIST_FACTOR * d0]
        if len(good) >= N:
            break
    else:
        raise RuntimeError(f"rattle_std={std}: no salen {N} configuraciones válidas; baja la amplitud")
    good = good[:N]

    u = np.array([a.positions - ref.positions for a in good])      # desplazamientos, forma (N, 126, 3)
    print(f"std={std:.3f}  |u| rms={np.sqrt((u**2).sum(2).mean()):.3f} Å  "
          f"max={np.linalg.norm(u, axis=2).max():.3f} Å  d_min={min(dmin(a) for a in good):.3f} Å")
    configs += [(std, a) for a in good]

print(f"\n{len(configs)} configuraciones")

## 4. Escribir las carpetas

Una carpeta por configuración, `cases/case_001` … `cases/case_200`, con:

- `input_scf.in`: la cabecera de arriba + `CELL_PARAMETERS` + `ATOMIC_POSITIONS` + `K_POINTS gamma`.
- `initial.xyz`: la misma estructura con su `rattle_std` (la usa `collect.py` para etiquetar el dataset).

Para ver una configuración: `from ase.visualize import view; view(configs[0][1])`.

In [ ]:
assert not Path("cases").exists(), "cases/ ya existe: bórralo o renómbralo antes de regenerar"

for i, (std, a) in enumerate(configs, start=1):
    d = Path("cases") / f"case_{i:03d}"
    d.mkdir(parents=True)

    a.info["rattle_std"] = std
    write(d / "initial.xyz", a)

    cell = "".join("%.10f %.10f %.10f\n" % tuple(v) for v in a.cell)
    pos = "".join("%-2s %.10f %.10f %.10f\n" % (s, *p) for s, p in zip(a.symbols, a.positions))
    (d / "input_scf.in").write_text(
        HEAD.format(nat=len(a)) + f"CELL_PARAMETERS angstrom\n{cell}ATOMIC_POSITIONS angstrom\n{pos}K_POINTS gamma\n")

print(f"{len(configs)} carpetas en cases/\n\nEn el cluster:  sbatch --array=1-{len(configs)}%20 job_scf.sh")

## 5. Siguiente paso

Copia esta carpeta entera al cluster (con `cases/`, `pseudos/`, `job_scf.sh`, `status.sh` y `collect.py`) y sigue el
`README.md`.